In [2]:
import pandas as pd

In [55]:
df_test = pd.read_csv('../aspects/extracted_aspects/all_beauty_absa_test_aspects.csv')
df_typo = pd.read_csv('../aspects/extracted_aspects/all_beauty_absa_typo_merges.csv')
df_vocab = pd.read_csv('../aspects/extracted_aspects/all_beauty_absa_vocab.csv')
df = pd.read_parquet('../../dataset/all_beauty.parquet')

In [57]:
print(df_test[df_test['review_idx'] == 2])

   review_idx  userId  itemId  raw_user  raw_item  cluster aspect sentiment  \
8           2   24976  103657     24976    103657    79389  smell  positive   

   polarity  mentions  confidence  pct_classifier  prob_positive  \
8  0.995978         1    0.996611             1.0       0.996611   

   prob_negative  prob_neutral  
8       0.000633      0.002756  


In [ ]:
# converters: o aspecto literal "nan" viraria NaN no parser padrão e quebraria o spaCy
df_aspects = pd.read_csv('../aspects/extracted_aspects/all_beauty_absa_test_aspects.csv',
                         converters={'aspect': str})

In [90]:
pd.set_option('display.max_columns', None)
print(df_aspects[df_aspects['userId'] == 0][['userId', 'itemId', 'aspect', 'sentiment', 'prob_positive', 'prob_neutral', 'prob_negative']])

        userId  itemId   aspect sentiment  prob_positive  prob_neutral  \
203850       0   19399    price  positive       0.997446      0.001660   
203851       0   19399  product  positive       0.998712      0.000884   

        prob_negative  
203850       0.000894  
203851       0.000403  


In [14]:
pares = df_aspects.groupby(['userId', 'itemId']).size()
print(pares.describe())                                  # aspectos por avaliação
print(pares.groupby(level=0).size().value_counts().head(10))  # avaliações por usuário

count    613795.000000
mean          2.718662
std           2.543881
min           1.000000
25%           1.000000
50%           2.000000
75%           3.000000
max         121.000000
dtype: float64
1     525761
2      28840
3       4280
4       1154
5        423
6        252
7        151
8        104
9         63
10        35
Name: count, dtype: int64


In [16]:
freq = df_aspects['aspect'].value_counts()
cum = freq.cumsum() / freq.sum()
for n in [50, 100, 200, 500, 1000, 5000]:
    top = set(freq.index[:n])
    cob_rev = df_aspects[df_aspects['aspect'].isin(top)].groupby(['userId','itemId']).ngroups / len(pares)
    print(n, f"menções={cum.iloc[n-1]:.3f}", f"avaliações={cob_rev:.3f}")
print("aspectos com 1 menção:", (freq == 1).sum())

50 menções=0.383 avaliações=0.674
100 menções=0.481 avaliações=0.759
200 menções=0.581 avaliações=0.825
500 menções=0.702 avaliações=0.894
1000 menções=0.780 avaliações=0.929
5000 menções=0.896 avaliações=0.971
aspectos com 1 menção: 64869


In [17]:
cum

aspect
hair                      0.033283
product                   0.063958
work                      0.090899
color                     0.117507
price                     0.140411
                            ...   
argan oil base product    0.999998
eye pack                  0.999998
goat milk body lotion     0.999999
package in plastic        0.999999
white stopper             1.000000
Name: count, Length: 95448, dtype: float64

In [35]:
print("NaN:", df_aspects['aspect'].isna().sum())
df_aspects['aspect_n'] = df_aspects['aspect'].str.lower().str.replace(r'[^\w\s]', ' ', regex=True).str.split().str.join(' ')
print("vazios após limpeza:", (df_aspects['aspect_n'] == '').sum())
df_aspects = df_aspects[df_aspects['aspect_n'].notna() & (df_aspects['aspect_n'] != '')].copy()

uniq = df_aspects['aspect_n'].unique()
lem_n = lambda d: " ".join(t.lemma_ if t.pos_ in ("NOUN", "PROPN") else t.text for t in d)
lem = {a: lem_n(d) for a, d in zip(uniq, nlp.pipe(uniq, batch_size=2000))}
df_aspects['aspect_n'] = df_aspects['aspect_n'].map(lem)

NaN: 0
vazios após limpeza: 0


In [36]:
import numpy as np
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer

freq = df_aspects['aspect_n'].value_counts()
enc = SentenceTransformer("all-MiniLM-L6-v2")
E = enc.encode(list(freq.index), batch_size=512, normalize_embeddings=True)
km = KMeans(n_clusters=150, n_init=5, random_state=42).fit(E[:5000])      # só a cabeça, sem pesos
C = km.cluster_centers_ / np.linalg.norm(km.cluster_centers_, axis=1, keepdims=True)
sim = E @ C.T
grp = np.where(sim.max(1) >= 0.6, sim.argmax(1), -1)
grp[:5000] = km.labels_
df_aspects['aspect_grp'] = df_aspects['aspect_n'].map(dict(zip(freq.index, grp)))

In [39]:
so_ruido = df_aspects.groupby(['userId', 'itemId'])['aspect_grp'].agg(lambda s: (s == -1).all())
print("avaliações só com aspectos -1:", round(so_ruido.mean(), 3))

avaliações só com aspectos -1: 0.016


In [40]:
m = df_aspects.groupby('aspect_grp')['aspect_n']
rot = pd.DataFrame({'menções': m.size(),
                    'termos': m.nunique(),
                    'top': m.agg(lambda s: ', '.join(s.value_counts().index[:10]))})
rot['tipo'] = ''          # preencher: topico | avaliativo | ruido
rot.sort_values('menções', ascending=False).to_csv('grupos_para_rotular.csv')

In [46]:
df_aspects.to_parquet('../aspect_based_recsys/aspects.parquet')